In [97]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import glob
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cv2

## Importing labels and images into variables

In [98]:
home = "images"
directories = "train", "test", "val"

In [99]:
def read_file_to_list(file_path):
    with open(file_path, 'r') as file:
        content = file.read().strip()
    return [int(char) for char in content]

In [100]:
y_train = read_file_to_list(f'{home}/{directories[0]}.txt')
y_test = read_file_to_list(f'{home}/{directories[1]}.txt')
y_val = read_file_to_list(f'{home}/{directories[2]}.txt')

In [101]:
train_folder = os.path.join(home, directories[0])
test_folder = os.path.join(home, directories[1])
val_folder = os.path.join(home, directories[2])

def get_image_content(folder_path):
    image_contents = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if os.path.isfile(file_path):
            image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            image_contents.append(image)
    return image_contents

x_train = get_image_content(train_folder)
x_test = get_image_content(test_folder)
x_val = get_image_content(val_folder)

## Training the CNN

In [102]:
x_train = np.array(x_train).reshape((-1, 224, 224, 1)).astype('float32')
x_val = np.array(x_val).reshape((-1, 224, 224, 1)).astype('float32')
x_test = np.array(x_test).reshape((-1, 224, 224, 1)).astype('float32')

y_train = tf.keras.utils.to_categorical(np.array(y_train), num_classes=4)
y_val = tf.keras.utils.to_categorical(np.array(y_val), num_classes=4)
y_test = tf.keras.utils.to_categorical(np.array(y_test), num_classes=4)

In [106]:
tf.random.set_seed(42)
model = tf.keras.Sequential()
input_shape = (224, 224, 1)
model.add(tf.keras.layers.Conv2D(10, 3, activation="relu", input_shape=input_shape))
model.add(tf.keras.layers.MaxPool2D())
model.add(tf.keras.layers.Conv2D(10, 3, activation="relu"))
model.add(tf.keras.layers.MaxPool2D())
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(4, activation="softmax"))

model.compile(loss=tf.keras.losses.CategoricalCrossentropy(),
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

history = model.fit(x=x_train, y=y_train, epochs=30, validation_data=(x_val, y_val), batch_size=16)

test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)

Epoch 1/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.2554 - loss: 101.1807 - val_accuracy: 0.2654 - val_loss: 1.5046
Epoch 2/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.2734 - loss: 1.4788 - val_accuracy: 0.2441 - val_loss: 1.4817
Epoch 3/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.2891 - loss: 1.4100 - val_accuracy: 0.2512 - val_loss: 1.4470
Epoch 4/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.3072 - loss: 1.3759 - val_accuracy: 0.2417 - val_loss: 1.4430
Epoch 5/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.3127 - loss: 1.3627 - val_accuracy: 0.2441 - val_loss: 1.4577
Epoch 6/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.3348 - loss: 1.3501 - val_accuracy: 0.2512 - val_loss: 1.4705
Epoch 7/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.3495 - loss: 1.3332 - val_accuracy: 0.2370 - val_loss: 1.4688
Epoch 8/30
211/211 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.3657 - loss: 1.3199 - val_ac

Beginning values look as if it was about to become an overfit model, will try to work on it.
Will do data augmentation and use dropout.